# Homework 3 — Longstaff-Schwartz : régression linéaire vs réseau de neurones

On compare deux versions de l'algorithme de Longstaff-Schwartz (LS) pour un put bermudéen dans Black-Scholes :

- **LS linéaire** : régression polynomiale (quadratique) de la continuation ;
- **LS non-linéaire** : réseau de neurones (MLP) pour la continuation.

L'objectif est d'étudier numériquement l'effet des réglages :
- nombre de scénarios ;
- nombre d'epochs ;
- taille de batch ;
- learning rate ;
- normalisation des données.

## Modèle et paramètres

Sous Black-Scholes (mesure risque-neutre),
\[
X_{t+\Delta t}=X_t\exp\!\left((r-	frac{1}{2}\sigma^2)\Delta t+\sigma\sqrt{\Delta t}\,Zight).
\]

Option bermudéenne exercable aux dates \(t_k=k/N\), \(k=0,\dots,N\).

Payoff donné dans l'énoncé :
\[
\phi_k(x)=e^{-rk/N}(K-x)_+.
\]

Paramètres imposés :
\[
r=0.1,\quad \sigma=0.25,\quad x_0=100,\quad K=110,\quad N=10,\quad T=1.
\]

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from sklearn.neural_network import MLPRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

In [ ]:
def simulate_bs_paths(x0, r, sigma, T, N, n_paths, seed=123):
    dt = T / N
    rng = np.random.default_rng(seed)

    z = rng.standard_normal((n_paths, N))
    X = np.empty((n_paths, N + 1), dtype=float)
    X[:, 0] = x0

    drift = (r - 0.5 * sigma**2) * dt
    vol = sigma * np.sqrt(dt)

    for k in range(N):
        X[:, k + 1] = X[:, k] * np.exp(drift + vol * z[:, k])

    return X


def payoff_discounted(x, k, K, r, dt):
    return np.exp(-r * k * dt) * np.maximum(K - x, 0.0)


def basis_poly2(x):
    x = np.asarray(x)
    return np.column_stack([np.ones_like(x), x, x**2])


def fit_linear_continuation(x_itm, y_itm):
    if x_itm.size == 0:
        return {"kind": "const", "c": 0.0}

    if np.unique(np.round(x_itm, 10)).size < 3:
        return {"kind": "const", "c": float(np.mean(y_itm))}

    A = basis_poly2(x_itm)
    beta, *_ = np.linalg.lstsq(A, y_itm, rcond=None)
    return {"kind": "poly2", "beta": beta}


def predict_linear_continuation(model, x):
    x = np.asarray(x)
    if model["kind"] == "poly2":
        y = basis_poly2(x) @ model["beta"]
    else:
        y = np.full_like(x, model["c"], dtype=float)
    return np.maximum(y, 0.0)

In [ ]:
def fit_nn_continuation(
    x_itm,
    y_itm,
    hidden_layer_sizes=(32, 32),
    learning_rate_init=1e-3,
    max_iter=200,
    batch_size=128,
    normalize=True,
    random_state=123,
):
    if x_itm.size == 0:
        return {"kind": "const", "c": 0.0}

    X_train = x_itm.reshape(-1, 1)

    mlp = MLPRegressor(
        hidden_layer_sizes=hidden_layer_sizes,
        activation="relu",
        solver="adam",
        learning_rate_init=learning_rate_init,
        max_iter=max_iter,
        batch_size=batch_size,
        random_state=random_state,
        early_stopping=False,
    )

    if normalize:
        model = Pipeline([
            ("scaler", StandardScaler()),
            ("mlp", mlp),
        ])
    else:
        model = mlp

    model.fit(X_train, y_itm)
    return {"kind": "nn", "model": model}


def predict_nn_continuation(model, x):
    x = np.asarray(x).reshape(-1, 1)

    if model["kind"] == "nn":
        y = model["model"].predict(x)
    else:
        y = np.full(x.shape[0], model["c"], dtype=float)

    return np.maximum(y, 0.0)

In [ ]:
def longstaff_schwartz_price(
    x0, K, r, sigma, T, N,
    n_paths=30000,
    seed=123,
    method="linear",
    nn_params=None,
):
    dt = T / N
    X = simulate_bs_paths(x0, r, sigma, T, N, n_paths, seed=seed)

    cashflow = payoff_discounted(X[:, N], N, K, r, dt)
    tau = np.full(n_paths, N, dtype=int)

    models = [None] * (N + 1)
    models[N] = {"kind": "terminal"}

    if nn_params is None:
        nn_params = {}

    for k in range(N - 1, -1, -1):
        immediate = payoff_discounted(X[:, k], k, K, r, dt)
        itm = immediate > 0.0

        x_itm = X[itm, k]
        y_itm = cashflow[itm]

        if method == "linear":
            model_k = fit_linear_continuation(x_itm, y_itm)
            cont = predict_linear_continuation(model_k, X[:, k])

        elif method == "nn":
            model_k = fit_nn_continuation(x_itm, y_itm, **nn_params)
            cont = predict_nn_continuation(model_k, X[:, k])

        else:
            raise ValueError("method doit être 'linear' ou 'nn'.")

        models[k] = model_k

        exercise_now = itm & (immediate >= cont)
        cashflow = np.where(exercise_now, immediate, cashflow)
        tau = np.where(exercise_now, k, tau)

    return {
        "price": float(np.mean(cashflow)),
        "models": models,
        "paths": X,
        "tau": tau,
        "dt": dt,
    }

In [ ]:
def continuation_on_grid(res, k, x_grid, method):
    model_k = res["models"][k]
    if k == len(res["models"]) - 1:
        return np.zeros_like(x_grid)

    if method == "linear":
        return predict_linear_continuation(model_k, x_grid)
    return predict_nn_continuation(model_k, x_grid)


def value_on_grid(res, k, x_grid, K, r, dt, method):
    phi = payoff_discounted(x_grid, k, K, r, dt)
    if k == len(res["models"]) - 1:
        return phi
    C = continuation_on_grid(res, k, x_grid, method)
    return np.maximum(phi, C)

## Expérience de base : comparaison LS linéaire vs LS réseau de neurones

In [ ]:
# Paramètres de l'énoncé
r = 0.1
sigma = 0.25
x0 = 100
K = 110
N = 10
T = 1.0

# Monte Carlo
n_paths = 30000
seed = 7

# Réglages NN de base
nn_base = dict(
    hidden_layer_sizes=(32, 32),
    learning_rate_init=1e-3,
    max_iter=200,
    batch_size=128,
    normalize=True,
    random_state=7,
)

res_linear = longstaff_schwartz_price(
    x0, K, r, sigma, T, N,
    n_paths=n_paths,
    seed=seed,
    method="linear",
)

res_nn = longstaff_schwartz_price(
    x0, K, r, sigma, T, N,
    n_paths=n_paths,
    seed=seed,
    method="nn",
    nn_params=nn_base,
)

print(f"Prix LS linéaire : {res_linear['price']:.6f}")
print(f"Prix LS réseau NN : {res_nn['price']:.6f}")
print(f"Écart (NN - linéaire) : {res_nn['price'] - res_linear['price']:.6f}")

In [ ]:
x_grid = np.linspace(40, 180, 400)

# Courbes de continuation C_k
fig, axes = plt.subplots(2, 5, figsize=(18, 6), sharex=True, sharey=True)
axes = axes.ravel()

for k in range(N):
    C_lin = continuation_on_grid(res_linear, k, x_grid, method="linear")
    C_nn = continuation_on_grid(res_nn, k, x_grid, method="nn")

    axes[k].plot(x_grid, C_lin, lw=2, label="linéaire")
    axes[k].plot(x_grid, C_nn, lw=2, ls="--", label="réseau NN")
    axes[k].set_title(f"k={k}")
    axes[k].grid(alpha=0.3)

axes[0].legend(fontsize=8)
fig.suptitle("Comparaison des continuations estimées C_k", fontsize=14)
fig.tight_layout()
plt.show()

In [ ]:
# Courbes de valeur V_k
fig, axes = plt.subplots(3, 4, figsize=(18, 10), sharex=True, sharey=True)
axes = axes.ravel()

for k in range(N + 1):
    V_lin = value_on_grid(res_linear, k, x_grid, K, r, res_linear["dt"], method="linear")
    V_nn = value_on_grid(res_nn, k, x_grid, K, r, res_nn["dt"], method="nn")

    axes[k].plot(x_grid, V_lin, lw=2, label="linéaire")
    axes[k].plot(x_grid, V_nn, lw=2, ls="--", label="réseau NN")
    axes[k].set_title(f"k={k}")
    axes[k].grid(alpha=0.3)

axes[-1].axis("off")
axes[0].legend(fontsize=8)
fig.suptitle("Comparaison des fonctions valeurs V_k", fontsize=14)
fig.tight_layout()
plt.show()

## Étude prospective des paramètres

On fait varier un paramètre à la fois autour d'un réglage de référence, et on compare le prix LS linéaire et le prix LS réseau de neurones.

In [ ]:
def run_benchmark_grid(
    x0, K, r, sigma, T, N,
    seeds,
    n_paths_list,
    epochs_list,
    batch_list,
    lr_list,
    norm_list,
):
    rows = []

    base_nn = dict(
        hidden_layer_sizes=(32, 32),
        learning_rate_init=1e-3,
        max_iter=200,
        batch_size=128,
        normalize=True,
        random_state=123,
    )

    # 1) Nombre de scénarios
    for n_paths in n_paths_list:
        for sd in seeds:
            pl = longstaff_schwartz_price(x0, K, r, sigma, T, N, n_paths=n_paths, seed=sd, method="linear")
            pn = longstaff_schwartz_price(x0, K, r, sigma, T, N, n_paths=n_paths, seed=sd, method="nn", nn_params=base_nn)
            rows.append({
                "study": "n_paths",
                "value": n_paths,
                "seed": sd,
                "linear": pl["price"],
                "nn": pn["price"],
            })

    # 2) Epochs
    for ep in epochs_list:
        nn_cfg = dict(base_nn)
        nn_cfg["max_iter"] = ep
        for sd in seeds:
            pl = longstaff_schwartz_price(x0, K, r, sigma, T, N, n_paths=30000, seed=sd, method="linear")
            pn = longstaff_schwartz_price(x0, K, r, sigma, T, N, n_paths=30000, seed=sd, method="nn", nn_params=nn_cfg)
            rows.append({
                "study": "epochs",
                "value": ep,
                "seed": sd,
                "linear": pl["price"],
                "nn": pn["price"],
            })

    # 3) Batch size
    for bs in batch_list:
        nn_cfg = dict(base_nn)
        nn_cfg["batch_size"] = bs
        for sd in seeds:
            pl = longstaff_schwartz_price(x0, K, r, sigma, T, N, n_paths=30000, seed=sd, method="linear")
            pn = longstaff_schwartz_price(x0, K, r, sigma, T, N, n_paths=30000, seed=sd, method="nn", nn_params=nn_cfg)
            rows.append({
                "study": "batch_size",
                "value": bs,
                "seed": sd,
                "linear": pl["price"],
                "nn": pn["price"],
            })

    # 4) Learning rate
    for lr in lr_list:
        nn_cfg = dict(base_nn)
        nn_cfg["learning_rate_init"] = lr
        for sd in seeds:
            pl = longstaff_schwartz_price(x0, K, r, sigma, T, N, n_paths=30000, seed=sd, method="linear")
            pn = longstaff_schwartz_price(x0, K, r, sigma, T, N, n_paths=30000, seed=sd, method="nn", nn_params=nn_cfg)
            rows.append({
                "study": "learning_rate",
                "value": lr,
                "seed": sd,
                "linear": pl["price"],
                "nn": pn["price"],
            })

    # 5) Normalisation
    for nm in norm_list:
        nn_cfg = dict(base_nn)
        nn_cfg["normalize"] = nm
        for sd in seeds:
            pl = longstaff_schwartz_price(x0, K, r, sigma, T, N, n_paths=30000, seed=sd, method="linear")
            pn = longstaff_schwartz_price(x0, K, r, sigma, T, N, n_paths=30000, seed=sd, method="nn", nn_params=nn_cfg)
            rows.append({
                "study": "normalization",
                "value": nm,
                "seed": sd,
                "linear": pl["price"],
                "nn": pn["price"],
            })

    return rows

In [ ]:
# Grilles de test (tailles raisonnables pour un notebook)
seeds = [3, 7, 11]
n_paths_list = [5000, 15000, 30000]
epochs_list = [50, 120, 250]
batch_list = [64, 128, 512]
lr_list = [1e-4, 1e-3, 5e-3]
norm_list = [False, True]

rows = run_benchmark_grid(
    x0=x0, K=K, r=r, sigma=sigma, T=T, N=N,
    seeds=seeds,
    n_paths_list=n_paths_list,
    epochs_list=epochs_list,
    batch_list=batch_list,
    lr_list=lr_list,
    norm_list=norm_list,
)

print(f"Nombre total d'expériences : {len(rows)}")

In [ ]:
def summarize_rows(rows):
    summary = {}
    for r_ in rows:
        key = (r_["study"], r_["value"])
        summary.setdefault(key, {"linear": [], "nn": []})
        summary[key]["linear"].append(r_["linear"])
        summary[key]["nn"].append(r_["nn"])

    out = []
    for (study, value), vals in summary.items():
        lin = np.array(vals["linear"])
        nn = np.array(vals["nn"])
        out.append({
            "study": study,
            "value": value,
            "linear_mean": lin.mean(),
            "linear_std": lin.std(ddof=1) if lin.size > 1 else 0.0,
            "nn_mean": nn.mean(),
            "nn_std": nn.std(ddof=1) if nn.size > 1 else 0.0,
            "diff_mean": (nn - lin).mean(),
        })
    return out


summary = summarize_rows(rows)

# Affichage texte propre
for study_name in ["n_paths", "epochs", "batch_size", "learning_rate", "normalization"]:
    print(f"\n=== Etude: {study_name} ===")
    sub = [d for d in summary if d["study"] == study_name]
    sub = sorted(sub, key=lambda x: str(x["value"]))
    for d in sub:
        print(
            f"value={d['value']!s:>6} | "
            f"lin={d['linear_mean']:.4f}±{d['linear_std']:.4f} | "
            f"nn={d['nn_mean']:.4f}±{d['nn_std']:.4f} | "
            f"(nn-lin)={d['diff_mean']:.4f}"
        )

In [ ]:
def plot_study(summary, study, title):
    sub = [d for d in summary if d['study'] == study]

    if study == 'normalization':
        x = np.arange(len(sub))
        labels = [str(d['value']) for d in sub]
    else:
        sub = sorted(sub, key=lambda x: float(x['value']))
        x = np.arange(len(sub))
        labels = [str(d['value']) for d in sub]

    lin_mean = [d['linear_mean'] for d in sub]
    lin_std = [d['linear_std'] for d in sub]
    nn_mean = [d['nn_mean'] for d in sub]
    nn_std = [d['nn_std'] for d in sub]

    plt.figure(figsize=(8, 4.5))
    plt.errorbar(x, lin_mean, yerr=lin_std, marker='o', capsize=4, label='LS linéaire')
    plt.errorbar(x, nn_mean, yerr=nn_std, marker='s', capsize=4, label='LS réseau NN')
    plt.xticks(x, labels)
    plt.title(title)
    plt.xlabel('Valeur du paramètre')
    plt.ylabel('Prix estimé')
    plt.grid(alpha=0.3)
    plt.legend()
    plt.tight_layout()
    plt.show()


plot_study(summary, 'n_paths', 'Impact du nombre de scénarios')
plot_study(summary, 'epochs', "Impact du nombre d'epochs (NN)")
plot_study(summary, 'batch_size', 'Impact de la taille de batch (NN)')
plot_study(summary, 'learning_rate', 'Impact du learning rate (NN)')
plot_study(summary, 'normalization', 'Impact de la normalisation (NN)')

## Discussion courte

Points généralement observés dans cette expérience :

- Quand le nombre de scénarios augmente, les deux méthodes se stabilisent (écart-type plus faible).
- Le réseau de neurones peut mieux s'adapter à une forme de continuation non-linéaire, mais il est plus sensible aux hyperparamètres.
- Le learning rate et la normalisation influencent fortement la stabilité de l'apprentissage NN.
- Trop peu d'epochs peut sous-apprendre, trop d'epochs peut sur-ajuster selon le bruit Monte Carlo.

Conclusion pratique : LS linéaire est plus robuste et rapide ; LS NN peut être plus flexible mais demande un réglage plus soigneux.